In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



In [2]:
sys.path.append("/root/work/tvm-ansor/gallery/constrained_gen_budget")
from modules.task_paths import load_and_register_tasks
tasks = load_and_register_tasks("/root/work/tvm-ansor/gallery/dataset/network_info_all")
# concrete_states = {}
# sketches_by_idx = {}
# for idx, task in enumerate(tasks):
#     concrete_state = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
#     for i, state in enumerate(concrete_state):
#         sketches_by_idx[idx] = (task.desc, f"{task.workload_key}_{i}")
#         concrete_states[f"{task.workload_key}_{i}"] = (task, state)

In [3]:
for idx, task in enumerate(tasks):
    print(f"Task {idx}: {task.desc}, {task.workload_key}")

Task 0: vm_mod_fused_nn_adaptive_avg_pool3d, ["1aa729c96f4afc0cf6bf84dff07364c6", [1, 18, 9, 1, 512], [1, 1, 1, 1, 512]]
Task 1: vm_mod_fused_nn_adaptive_avg_pool3d, ["1aa729c96f4afc0cf6bf84dff07364c6", [2, 18, 9, 1, 512], [2, 1, 1, 1, 512]]
Task 2: vm_mod_fused_nn_adaptive_avg_pool3d, ["1aa729c96f4afc0cf6bf84dff07364c6", [4, 18, 9, 1, 512], [4, 1, 1, 1, 512]]
Task 3: vm_mod_fused_nn_adaptive_avg_pool2d_2, ["25252ef28760d56401943904a46661f3", [1, 16, 16, 480], [1, 1, 1, 480]]
Task 4: vm_mod_fused_nn_adaptive_avg_pool2d_3, ["25252ef28760d56401943904a46661f3", [1, 16, 16, 672], [1, 1, 1, 672]]
Task 5: vm_mod_fused_nn_adaptive_avg_pool2d_2, ["25252ef28760d56401943904a46661f3", [4, 16, 16, 480], [4, 1, 1, 480]]
Task 6: vm_mod_fused_nn_adaptive_avg_pool2d_3, ["25252ef28760d56401943904a46661f3", [4, 16, 16, 672], [4, 1, 1, 672]]
Task 7: vm_mod_fused_nn_adaptive_avg_pool2d_2, ["25252ef28760d56401943904a46661f3", [8, 16, 16, 480], [8, 1, 1, 480]]
Task 8: vm_mod_fused_nn_adaptive_avg_pool2d_3, 

In [4]:
ENABLED_CONSTRAINTS_NO_VECTORIZE = (
    'shared_memory',
    'max_vector_bytes',
    'max_threads',
    'max_vthread',
    'innermost_split',
    'split_structure',
)

sys.path.append("/root/work/tvm-ansor/gallery/constrained_gen_budget")
from modules.symbolic_state_bridge import build_symbolic_state
from modules.schedule_generator import ScheduleGenerator
task_idx = 1490
task = tasks[task_idx]
sample = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()[0]
sym_state = build_symbolic_state(task, sample)
sg = ScheduleGenerator(sym_state, task=task, base_state=sample)
print(task_idx, ":", task.desc)
print(task.workload_key)
print(sym_state)
# cs.transform_steps[-5].extent

1490 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform
["3eda1939e30b947e921f5e1814346365", [1, 56, 56, 128], [6, 6, 32, 128], [1, 56, 56, 32]]
Placeholder: p0, p1
data_pack auto_unroll: ur_63
blockIdx.x p.0@ci.0@p.1@ci.1@.0 (0,ceil(ceil(196/(min(sp_27_0,196)))*ceil(128/(min(sp_28_0,128)))*min(sp_27_0,196)*min(sp_28_0,128)/(min(sp_60_0,ceil(196/(min(sp_27_0,196)))*ceil(128/(min(sp_28_0,128)))*min(sp_27_0,196)*min(sp_28_0,128)))))
  threadIdx.x p.0@ci.0@p.1@ci.1@.1 (0,min(sp_60_0,ceil(196/(min(sp_27_0,196)))*ceil(128/(min(sp_28_0,128)))*min(sp_27_0,196)*min(sp_28_0,128)))
    for eps (0,6)
      for nu (0,6)
        for p (0,1)
          for ci (0,1)
            input_tile = ...
    unroll eps (0,6)
      unroll nu (0,6)
        unroll r_a (0,6)
          unroll r_b (0,6)
            data_pack = ...
blockIdx.x eps.0@nu.0@p.0@co.0@ (0,ceil(ceil(ceil(6/(min(sp_9_2*sp_9_3,6)))/(min(sp_9_1,ceil(6/(min(sp_9_2*sp_9_3,6))))))/(min(sp_9_0,ceil(ceil(6/(min(sp_9_2*sp_9_3,6)))/(mi

In [ ]:
sg.s._exception_split_names

In [5]:
print(sg._get_constraints_str())

[grid_1: blockIdx.x@s4.i0]
  max_threads: min(sp_60_0,ceil(196/(min(sp_27_0,196)))*ceil(128/(min(sp_28_0,128)))*min(sp_27_0,196)*min(sp_28_0,128)) <= 1024

[grid_0: blockIdx.x@s9.i0]
  max_threads:
    min(sp_9_1,ceil(6/(min(sp_9_2*sp_9_3,6))))
    * min(sp_10_1,ceil(6/(min(sp_10_2*sp_10_3,6))))
    * min(sp_11_1,ceil(196/(min(sp_11_2*sp_11_3,196))))
    * min(sp_12_1,ceil(32/(min(sp_12_2*sp_12_3,32))))
    <= 1024
  max_vthread:
    min(sp_9_0,ceil(ceil(6/(min(sp_9_2*sp_9_3,6)))/(min(sp_9_1,ceil(6/(min(sp_9_2*sp_9_3,6)))))))
    * min(sp_10_0,ceil(ceil(6/(min(sp_10_2*sp_10_3,6)))/(min(sp_10_1,ceil(6/(min(sp_10_2*sp_10_3,6)))))))
    * min(sp_11_0,ceil(ceil(196/(min(sp_11_2*sp_11_3,196)))/(min(sp_11_1,ceil(196/(min(sp_11_2*sp_11_3,196)))))))
    * min(sp_12_0,ceil(ceil(32/(min(sp_12_2*sp_12_3,32)))/(min(sp_12_1,ceil(32/(min(sp_12_2*sp_12_3,32)))))))
    <= 8
  shared_memory:
    min(ceil(128 / sp_13_1), sp_13_0)
    * min(128, sp_13_1)
    * ((sp_9_1 * (sp_9_0 * (ceil(ceil(ceil(ceil((c

In [10]:
sg._get_var_order_phase_entries()

[{'phase_name': 'grid_0__memory_split_structure',
  'phase_family': 'memory_split_structure',
  'grid_scope': (),
  'param_names': ['sp_1_3',
   'sp_1_2',
   'sp_1_1',
   'sp_1_0',
   'sp_2_3',
   'sp_2_2',
   'sp_2_1',
   'sp_2_0',
   'sp_3_3',
   'sp_3_2',
   'sp_3_1',
   'sp_3_0',
   'sp_4_3',
   'sp_4_2',
   'sp_4_1',
   'sp_4_0',
   'sp_5_1',
   'sp_5_0',
   'sp_6_1',
   'sp_6_0'],
  'param_entries': [{'param_name': 'sp_1_3',
    'param_kind': 'split',
    'split_step_idx': 1,
    'split_position': 3,
    'split_extent': 1,
    'split_group_param_names': ['sp_1_0', 'sp_1_1', 'sp_1_2', 'sp_1_3'],
    'collapsed_factor_param_names': ['sp_1_0', 'sp_1_1', 'sp_1_2'],
    'is_innermost': True},
   {'param_name': 'sp_1_2',
    'param_kind': 'split',
    'split_step_idx': 1,
    'split_position': 2,
    'split_extent': 1,
    'split_group_param_names': ['sp_1_0', 'sp_1_1', 'sp_1_2', 'sp_1_3'],
    'collapsed_factor_param_names': ['sp_1_0', 'sp_1_1'],
    'is_innermost': False},
   {'param

In [11]:
sg.get_full_var_order_entries()

{'phase_count': 2,
 'param_order': ['sp_1_3',
  'sp_1_2',
  'sp_1_1',
  'sp_1_0',
  'sp_2_3',
  'sp_2_2',
  'sp_2_1',
  'sp_2_0',
  'sp_3_3',
  'sp_3_2',
  'sp_3_1',
  'sp_3_0',
  'sp_4_3',
  'sp_4_2',
  'sp_4_1',
  'sp_4_0',
  'sp_5_1',
  'sp_5_0',
  'sp_6_1',
  'sp_6_0',
  'sp_26_0',
  'sp_31_0',
  'ur_35'],
 'phases': [{'phase_name': 'grid_0__memory_split_structure',
   'phase_family': 'memory_split_structure',
   'grid_scope': (),
   'param_names': ['sp_1_3',
    'sp_1_2',
    'sp_1_1',
    'sp_1_0',
    'sp_2_3',
    'sp_2_2',
    'sp_2_1',
    'sp_2_0',
    'sp_3_3',
    'sp_3_2',
    'sp_3_1',
    'sp_3_0',
    'sp_4_3',
    'sp_4_2',
    'sp_4_1',
    'sp_4_0',
    'sp_5_1',
    'sp_5_0',
    'sp_6_1',
    'sp_6_0'],
   'param_entries': [{'param_name': 'sp_1_3',
     'param_kind': 'split',
     'split_step_idx': 1,
     'split_position': 3,
     'split_extent': 1,
     'split_group_param_names': ['sp_1_0', 'sp_1_1', 'sp_1_2', 'sp_1_3'],
     'collapsed_factor_param_names': ['sp

In [13]:
sg.get_constraints_under_assignment()

{'assignment': {'params': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1}},
 'domains': {'all': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1,
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
   'sp_6_0': [1, 3],
   'sp_6_1': [1, 3],
   'sp_26_0': [1, 36],
   'sp_31_0': [1, 64]},
  'fixed': {'sp_1_0': 1, 'sp_1_1': 1, 'sp_1_2': 1, 'sp_1_3': 1},
  'remaining': {'sp_26_0': [1, 36],
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_31_0': [1, 64],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
 

In [4]:
sg.get_param_candidates(name='sp_31_0')

{'assignment': {'params': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1}},
 'domains': {'all': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1,
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
   'sp_6_0': [1, 3],
   'sp_6_1': [1, 3],
   'sp_26_0': [1, 36],
   'sp_31_0': [1, 60]},
  'fixed': {'sp_1_0': 1, 'sp_1_1': 1, 'sp_1_2': 1, 'sp_1_3': 1},
  'remaining': {'sp_26_0': [1, 36],
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_31_0': [1, 60],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
 